# ClinGuard-DiLoCo: Kaggle session 4 (gate G3: can the model see the findings?)

Session 3: no setting beat a model that always writes "normal", and every model predicted a study's report
as well from **another patient's X-ray** as from its own. Sessions 2-3 also quantized the **vision tower** to
4 bit (bitsandbytes' default quantizes every layer except the output layer). This session checks whether
that is what blinds the model, and retrains with the vision tower in fp16 at the same time:

| GPU | Jobs, in order | Estimate |
|---|---|---|
| 0 | **vision check**: the untrained model with a 4-bit vision tower, an fp16 vision tower, and no quantization at all, asked to prefer "The heart is enlarged." over "The heart size is normal." (and 3 more findings) on the development studies; then retrain site A (`g3_A_lr2e4_visfp16`) | ~25 min + ~1.5 h |
| 1 | retrain site B with the fp16 vision tower (`g3_B_lr2e4_visfp16`) | ~1.5 h |
| 0 | image test of the two new models | ~10 min |

Training uploads to the private HF repo every ~15 min with a lease; after a crash, run the notebook again and
each job continues where it stopped (finished jobs are skipped).

**Settings** (right-hand panel): GPU **T4 x2**, Internet **on**, input dataset **raddar/chest-xrays-indiana-university**,
secret **HF_TOKEN** attached. Keep the notebook **private**.

**How to run:** *Save Version* -> **Save & Run All (Commit)**, then close the tab. A commit run keeps going
for up to 12 h with the browser closed; an interactive session in the editor is switched off when idle.
When the version has finished, open its *Logs* / *Output* and send the output of the last two cells.

In [ ]:
HF_REPO = "tasbid66/clinguard-runs"
GIT_REF = "main"     # a resumed run refuses a different commit; to resume after main moved, set the commit printed in its log
import glob, os, subprocess, sys, time, json
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Download the 3B model once, so the two GPUs do not race for it.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ClinGuard/src")
from huggingface_hub import snapshot_download
from clinguard.model import load
snapshot_download(load.MODEL_ID, revision=load.MODEL_REVISION)
print("model cached")

## Launch (one queue per GPU, in the background)

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
os.makedirs("/kaggle/working/logs", exist_ok=True)
RUN = f"python scripts/run_job.py --hf-repo {HF_REPO} --work-dir /kaggle/working/runs --config configs/experiment"
CHECK = f"python scripts/vision_check.py --hf-repo {HF_REPO} --name session4 --work-dir /kaggle/working/runs"
queues = {0: [("vision check", CHECK), ("g3_A_lr2e4_visfp16", f"{RUN}/g3_A_lr2e4_visfp16.yaml")],
          1: [("g3_B_lr2e4_visfp16", f"{RUN}/g3_B_lr2e4_visfp16.yaml")]}
procs = {}
for gpu, jobs in queues.items():
    log = open(f"/kaggle/working/logs/gpu{gpu}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    # ';' so a later job still runs if an earlier one fails
    procs[gpu] = subprocess.Popen(" ; ".join(cmd for _, cmd in jobs), shell=True, stdout=log,
                                  stderr=subprocess.STDOUT, env=env)
    print("GPU", gpu, "started:", " -> ".join(label for label, _ in jobs))

## Wait, printing progress every 5 minutes

In [ ]:
import time
def tail(path, n=3):
    lines = [l for l in open(path, errors="replace").read().splitlines()
             if l.strip() and "Warning" not in l and "warn" not in l]
    return lines[-n:]
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {int(time.time() - start) // 60} min")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):", *tail(f"/kaggle/working/logs/gpu{gpu}.log"), sep="\n   ")
print({gpu: p.returncode for gpu, p in procs.items()})

## Image test of the two retrained models

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
names = ["g3_A_lr2e4_visfp16", "g3_B_lr2e4_visfp16"]
cmd = (f"python scripts/image_test.py --hf-repo {HF_REPO} --name session4 --site A --work-dir /kaggle/working/runs "
       "--configs " + " ".join(f"configs/experiment/{n}.yaml" for n in names))
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
open("/kaggle/working/logs/image_test.log", "w").write(r.stdout + r.stderr)
print("\n".join(l for l in r.stdout.splitlines() if "Warning" not in l)[-4000:])
if r.returncode:
    print("image test failed:", r.stderr[-3000:])

## Summary (self-contained: after a restart, run cell 2 and then this cell)

In [ ]:
import importlib.util, json, os, sys
from huggingface_hub import hf_hub_download
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")
import run_job as rj
import vision_check
def fetch(path):
    try:
        return hf_hub_download(HF_REPO, path, force_download=True)
    except Exception as e:
        print("missing:", path, type(e).__name__); return None
v = fetch("diagnostics/vision_check_session4.json")
if v:
    vision_check.print_report(json.load(open(v)))
print()
for n in ("g3_A_lr2e4_visfp16", "g3_B_lr2e4_visfp16"):
    rid = rj.run_id_for(rj.load_config(f"configs/experiment/{n}.yaml"))
    lease = fetch(f"runs/{rid}/LEASE.json")
    print(f"{n:20s} {rid}  status: {json.load(open(lease))['status'] if lease else '-'}")
    r = fetch(f"runs/{rid}/rounds.jsonl")
    if r:
        rows = [json.loads(l) for l in open(r)]
        site = next(iter(rows[0]["train_loss"]))
        print("   train loss by round:", " ".join(f"{x['train_loss'][site]:.3f}" for x in rows),
              f"| {rows[-1]['cost'][site]['train_sec'] / 100:.2f} s/step")
t = fetch("diagnostics/image_test_session4.json")
if t:
    t = json.load(open(t))
    print(f"\nimage test, site {t['site']} {t['role']}, {t['n']} studies (loss of the reference report; lower = better)")
    for label, r in t["results"].items():
        a, b = r["all"], r["abnormal"]
        print(f"  {label[:52]:52s} true {a['loss_true']:.3f} other {a['loss_shuffled']:.3f} blank {a['loss_blank']:.3f}"
              f" | abnormal: true {b['loss_true']:.3f} other {b['loss_shuffled']:.3f}"
              f" true better in {b['share_true_image_better']:.0%}")

**Send Tasbid / Claude:** the output of the last two cells. Generated reports stay in the HF repo; they are
scored locally in the evaluation environment.